# 00 · Análisis exploratorio del dataset

Corpus ya scrapeado, filtrado, deduplicado y dividido por `DatasetInsectos`
(`manifiesto.json`, semilla 42, reparto por observación). Este notebook solo lo
inspecciona: nunca recalcula el split ni toca las imágenes.

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)

print("Raíz del proyecto:", ROOT)

In [ ]:
import pandas as pd

from scripts import corpus
from scripts.vizStyle import SERIE_1, SERIE_2, SERIE_3, applyStyle, limpiarEje

corpus.materializar()
manifiesto = corpus.manifiesto()
print(manifiesto["semilla"], manifiesto["num_clases"], manifiesto["conteos"])

## 1 · Conteo por especie y por split

In [ ]:
reparto = manifiesto["reparto"]
filas = [
    {"especie": especie, **{s: len(archivos[s]) for s in ("train", "test", "validate")}}
    for especie, archivos in reparto.items()
]
conteos = pd.DataFrame(filas)
conteos["total"] = conteos[["train", "test", "validate"]].sum(axis=1)
conteos = conteos.sort_values("total", ascending=False).reset_index(drop=True)
conteos.describe()

In [ ]:
print(f"{len(conteos)} especies, {conteos['total'].sum()} imágenes totales")
conteos[["train", "test", "validate", "total"]].sum().to_frame("imagenes")

## 2 · Balance del corpus

Con cuántas imágenes queda la especie más escasa y la más abundante — define
qué tanto pesa `class_weight="balanced"` en el entrenamiento.

In [ ]:
applyStyle()
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, max(4, 0.09 * len(conteos))))
ax.barh(range(len(conteos)), conteos["total"], color=SERIE_1, height=0.75)
ax.set_yticks([])
ax.set(xlabel="Imágenes", title="Imágenes por especie (ordenado, todas las 79)")
ax.axvline(conteos["total"].median(), color=SERIE_2, linestyle="--", linewidth=1.2)
fig.tight_layout()
plt.show()

In [ ]:
print("Especie con más imágenes:")
print(conteos.iloc[0][["especie", "total"]].to_string())
print("\nEspecie con menos imágenes:")
print(conteos.iloc[-1][["especie", "total"]].to_string())

## 3 · Especies de cola larga (< 40 imágenes)

In [ ]:
colaLarga = conteos[conteos["total"] < 40].sort_values("total")
print(f"{len(colaLarga)} especies por debajo de 40 imágenes")
colaLarga[["especie", "train", "test", "validate", "total"]]

## 4 · Proporción del split por especie

El split es 70/20/10 combinando semilla + nombre de especie; se verifica que
se cumpla razonablemente incluso en las especies más escasas.

In [ ]:
proporciones = conteos.copy()
for s in ("train", "test", "validate"):
    proporciones[f"{s}_pct"] = proporciones[s] / proporciones["total"]
proporciones[["especie", "total", "train_pct", "test_pct", "validate_pct"]].describe()

In [ ]:
applyStyle()
fig, ax = plt.subplots(figsize=(6, 4.5))
for col, color, label in (
    ("train_pct", SERIE_1, "train"),
    ("test_pct", SERIE_2, "test"),
    ("validate_pct", SERIE_3, "validate"),
):
    ax.hist(proporciones[col], bins=20, color=color, alpha=0.6, label=label)
ax.set(xlabel="Proporción del split", ylabel="Especies", title="Distribución de proporciones por split")
ax.legend()
fig.tight_layout()
plt.show()

## 5 · Ejemplos por especie

In [ ]:
import random

from PIL import Image

random.seed(42)
muestraEspecies = random.sample(sorted(reparto.keys()), 6)

applyStyle()
fig, ejes = plt.subplots(1, len(muestraEspecies), figsize=(15, 3))
for ax, especie in zip(ejes, muestraEspecies):
    archivo = reparto[especie]["train"][0]
    img = Image.open(Path("dataset/train") / especie / archivo)
    ax.imshow(img)
    limpiarEje(ax)
    ax.set_title(especie.replace("_", " "), fontsize=8, style="italic")
fig.tight_layout()
plt.show()

## 6 · Tamaño de imagen

In [ ]:
tamanos = set()
for especie in muestraEspecies:
    for archivo in reparto[especie]["train"][:5]:
        img = Image.open(Path("dataset/train") / especie / archivo)
        tamanos.add(img.size)
print("Tamaños encontrados en la muestra:", tamanos)

## Resumen

- 35 clases (con *Atta cephalotes* y *A. colombica* fusionadas en `atta_spp`),
  split 70/20/10 ya armado por `DatasetInsectos` (semilla 42).
- Imágenes preprocesadas a 448×448 RGB — el pipeline de entrenamiento las
  reescala a 320×320
- El corpus está desbalanceado (ver especies de cola larga); todos los
  entrenamientos usan `class_weight="balanced"` para compensarlo.